In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp,col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


In [0]:
sourceflightURL='http://api.aviationstack.com/v1/flights'

api_key = dbutils.secrets.get(
    catalog="flightdata",
    schema="bronze",
    key="api_key"
)
volume_path ='/Volumes/flightdata/bronze/flight_data'

In [0]:
sourceFlightData=requests.get(sourceflightURL, params={'access_key': api_key}).json()['data']


In [0]:
jsonSourceFlightData =json.dumps(sourceFlightData)

In [0]:

flight_schema = StructType([
    StructField("flight_date", StringType(), True),
    StructField("flight_status", StringType(), True),
    StructField("departure", StructType([
        StructField("airport", StringType(), True),
        StructField("timezone", StringType(), True),
        StructField("iata", StringType(), True),
        StructField("icao", StringType(), True),
        StructField("terminal", StringType(), True),
        StructField("gate", StringType(), True),
        StructField("delay", IntegerType(), True),
        StructField("scheduled", StringType(), True),
        StructField("estimated", StringType(), True),
        StructField("actual", StringType(), True),
        StructField("estimated_runway", StringType(), True),
        StructField("actual_runway", StringType(), True),
    ]), True),
    StructField("arrival", StructType([
        StructField("airport", StringType(), True),
        StructField("timezone", StringType(), True),
        StructField("iata", StringType(), True),
        StructField("icao", StringType(), True),
        StructField("terminal", StringType(), True),
        StructField("gate", StringType(), True),
        StructField("baggage", StringType(), True),
        StructField("scheduled", StringType(), True),
        StructField("delay", IntegerType(), True),
        StructField("estimated", StringType(), True),
        StructField("actual", StringType(), True),
        StructField("estimated_runway", StringType(), True),
        StructField("actual_runway", StringType(), True),
    ]), True),
    StructField("airline", StructType([
        StructField("name", StringType(), True),
        StructField("iata", StringType(), True),
        StructField("icao", StringType(), True),
    ]), True),
    StructField("flight", StructType([
        StructField("number", StringType(), True),
        StructField("iata", StringType(), True),
        StructField("icao", StringType(), True),
        StructField("codeshared", StringType(), True),
    ]), True),
    StructField("aircraft", StructType([
        StructField("registration", StringType(), True),
        StructField("iata", StringType(), True),
        StructField("icao", StringType(), True),
        StructField("icao24", StringType(), True),
    ]), True),
    StructField("live", StringType(), True),
])

sourceFlightDF = spark.createDataFrame(sourceFlightData, schema=flight_schema)

In [0]:
(sourceFlightDF
    .write
    .mode("overwrite")
    .partitionBy("flight_date")
    .json(volume_path))

In [0]:
sourceFlightDF.display()

In [0]:
sourceFlightDF = sourceFlightDF.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
(sourceFlightDF
    .write
    .mode("overwrite")
    .partitionBy("flight_date")
    .json(volume_path))

In [0]:
(sourceFlightDF
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable("flightdata.bronze.bronze_flights"))

In [0]:
display(spark.table('flightdata.bronze.bronze_flights').limit(5))